# Day 2 — Cameras, Size, and Seeing in 3D

Yesterday you wrote your first Python. Tomorrow you drop a 3D object into your own video.
Today is the part in between: **how a camera turns a 3D world into a flat picture, and how
a computer works backwards.**

0. **Vectors** — a short primer
1. **Where is the camera?** — what a pose is, and why a photo cannot tell you size
2. **From photos to a reconstruction** — finding points, matching, solving for a camera
3. **Putting something in** — choosing a size, standing it on the ground

**First:** `Runtime > Change runtime type > GPU`, then run the two setup cells below.

In [ ]:
# @title Setup: install and find the data (run me first)
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'panda3d'], check=True)

# Where the workshop clips and the precomputed reconstruction live: one zip in the repo.
REPO_RAW = 'https://raw.githubusercontent.com/prash-red/cs_academy_vfx/main'
ARCHIVE = 'day2_data.zip'


def find_data_dir():
    """Uses the repo's own data folder when the notebook runs inside a checkout, and
    otherwise downloads the day's data as a single zip and unpacks it."""
    for candidate in (Path('data'), Path('../data'), Path('../../data'),
                      Path('/content/cs_academy_vfx/data')):
        if (candidate / 'boulder.mp4').exists():
            return candidate.resolve()

    local = Path('workshop_data')
    if not (local / 'boulder.mp4').exists():
        local.mkdir(parents=True, exist_ok=True)
        print(f'downloading {ARCHIVE} ...')
        urllib.request.urlretrieve(f'{REPO_RAW}/data/{ARCHIVE}', local / ARCHIVE)
        with zipfile.ZipFile(local / ARCHIVE) as archive:
            archive.extractall(local)
        print(f'unpacked {ARCHIVE}')
    return local.resolve()


DATA = find_data_dir()
print(f'Using data from {DATA}')

In [ ]:
# @title Setup: the 3D scene and the helpers (run me second)
import json

# Draw figures into the notebook rather than into a window. Colab does this anyway;
# elsewhere the default can be a GUI backend, where plt.show() blocks forever waiting for
# a window that never appears.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except (NameError, AttributeError):
    pass

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)

VIEW_W, VIEW_H = 640, 360
VIEW_FOCAL = 480.0   # a wide-ish view, so something is almost always in frame

# ---------------------------------------------------------------- the maths bit
# Everything below uses the same convention as tomorrow's notebook:
# X = right, Y = down, Z = forward, and a pose is (position, forward, up).

WORLD_UP = np.array([0.0, -1.0, 0.0])


def normalize(v):
    """A vector pointing the same way, but with length exactly 1."""
    return np.asarray(v, dtype=float) / np.linalg.norm(v)


def camera_axes(forward, up):
    """A camera's own three directions. right is at right angles to both the others --
    that is what the cross product gives you."""
    forward = normalize(forward)
    up = normalize(up)
    right = normalize(np.cross(forward, up))
    up = np.cross(right, forward)      # re-square `up` in case it was tilted
    return right, up, forward


def look_at(position, target, up=WORLD_UP):
    """The pose of a camera standing at `position` and looking at `target`."""
    forward = normalize(np.asarray(target, float) - np.asarray(position, float))
    right, up, forward = camera_axes(forward, up)
    return dict(position=np.asarray(position, float), forward=forward, up=up)


def pose_from_angles(position, turn=0.0, tilt=0.0, roll=0.0):
    """A pose built the way a game does it: stand somewhere, then turn, tilt and roll.
    Moving the position leaves the aim untouched."""
    turn, tilt, roll = np.radians([turn, tilt, roll])
    forward = np.array([np.sin(turn) * np.cos(tilt), -np.sin(tilt),
                        np.cos(turn) * np.cos(tilt)])
    right = normalize(np.cross(forward, WORLD_UP))
    up = np.cross(right, forward)
    up = up * np.cos(roll) + np.cross(forward, up) * np.sin(roll)
    return dict(position=np.asarray(position, float), forward=normalize(forward),
                up=normalize(up))


def angles_towards(position, target=(0.0, 0.0, 0.0)):
    """The `turn` and `tilt` that make a camera at `position` look at `target` -- so a
    target pose can be written down by saying where it stands and what it looks at."""
    forward = normalize(np.asarray(target, float) - np.asarray(position, float))
    tilt = np.degrees(np.arcsin(-forward[1]))
    turn = np.degrees(np.arctan2(forward[0], forward[2]))
    return turn, tilt


def rotation_of(pose):
    """The 3x3 whose rows are the camera's own axes, as day 3 stores them."""
    right, up, forward = camera_axes(pose['forward'], pose['up'])
    return np.stack([right, -up, forward])


def project(pose, points, focal, width, height):
    """Where each 3D point lands in this camera's picture, in pixels."""
    R = rotation_of(pose)
    local = (np.atleast_2d(points) - np.asarray(pose['position'], float)) @ R.T
    in_front = local[:, 2] > 1e-6
    depth = np.where(in_front, local[:, 2], 1.0)
    pixels = np.stack([focal * local[:, 0] / depth + width / 2,
                       focal * local[:, 1] / depth + height / 2], axis=1)
    return pixels, in_front


def pose_error(pose_a, pose_b):
    """How far apart two poses are: distance of position, degrees of aim."""
    distance = np.linalg.norm(np.asarray(pose_a['position'], float) -
                              np.asarray(pose_b['position'], float))
    cos = np.clip(np.dot(normalize(pose_a['forward']), normalize(pose_b['forward'])), -1, 1)
    return distance, np.degrees(np.arccos(cos))


# ------------------------------------------------------------- the renderer bit
from panda3d.core import loadPrcFileData

loadPrcFileData('', 'window-type offscreen')
loadPrcFileData('', 'audio-library-name null')
loadPrcFileData('', 'depth-bits 24')
loadPrcFileData('', f'win-size {VIEW_W} {VIEW_H}')

from direct.showbase.ShowBase import ShowBase
from panda3d.core import (AmbientLight, DirectionalLight, GeomVertexReader, Point3,
                          SamplerState, TextureStage, Vec3, Vec4)

SKY_COLOR = Vec4(0.62, 0.72, 0.85, 1)


def cv_to_panda(v):
    """Our (X-right, Y-down, Z-forward) -> Panda3D's (X-right, Y-forward, Z-up)."""
    x, y, z = v
    return Vec3(float(x), float(z), float(-y))


def panda_to_cv(v):
    """The way back."""
    return np.array([v[0], -v[2], v[1]], dtype=float)


class Renderer(ShowBase):
    """One offscreen 3D view we can point anywhere. A frame takes a few milliseconds, so
    sliders can redraw live."""

    def __init__(self):
        super().__init__()
        self.win.setClearColor(SKY_COLOR)
        ambient = AmbientLight('ambient')
        ambient.setColor(Vec4(0.45, 0.45, 0.48, 1))
        self.render.setLight(self.render.attachNewNode(ambient))
        key = DirectionalLight('key')
        key.setColor(Vec4(0.85, 0.83, 0.78, 1))
        key_np = self.render.attachNewNode(key)
        key_np.setHpr(-35, -50, 0)
        self.render.setLight(key_np)
        self.scene_root = self.render.attachNewNode('scene')
        self.object_root = self.render.attachNewNode('objects')
        self.object_root.hide()
        self.disableMouse()

    def look_from(self, pose, focal=VIEW_FOCAL):
        position = cv_to_panda(pose['position'])
        forward = cv_to_panda(pose['forward'])
        up = cv_to_panda(pose['up'])
        self.camera.setPos(Point3(*position))
        self.camera.lookAt(Point3(*(position + forward)), up)
        self.camLens.setFilmSize(VIEW_W, VIEW_H)
        self.camLens.setFocalLength(focal)
        self.camLens.setAspectRatio(VIEW_W / VIEW_H)

    def image(self):
        self.graphicsEngine.renderFrame()
        self.graphicsEngine.renderFrame()
        tex = self.win.getScreenshot()
        data = tex.getRamImageAs('RGB')
        array = np.frombuffer(data, dtype=np.uint8).reshape(
            (tex.getYSize(), tex.getXSize(), 3))
        return np.flipud(array).copy()

    def render_from(self, pose, focal=VIEW_FOCAL):
        self.look_from(pose, focal)
        return self.image()


renderer = Renderer()


def fit_to_unit(node):
    """Rescales and shifts a model so its largest side is 1 and its middle is at zero."""
    bounds = node.getTightBounds()
    if bounds is None:
        return 1.0
    low, high = bounds
    size = max((high - low).x, (high - low).y, (high - low).z, 1e-6)
    node.setScale(1.0 / size)
    node.setPos(-(low + high) * 0.5 / size)
    return float(size)


def model_points(node, limit=2500, seed=0):
    """A point cloud of a model's own surface, in our coordinates. Lets us draw a model in
    a plain 3D plot, without a renderer."""
    points = []
    own = node.getTransform().getMat()   # the fit_to_unit scale/shift lives here
    for geom_np in node.findAllMatches('**/+GeomNode'):
        matrix = geom_np.getMat(node) * own
        for index in range(geom_np.node().getNumGeoms()):
            reader = GeomVertexReader(geom_np.node().getGeom(index).getVertexData(),
                                      'vertex')
            while not reader.isAtEnd():
                points.append(panda_to_cv(matrix.xformPoint(reader.getData3f())))
    points = np.array(points)
    if len(points) > limit:
        points = points[np.random.default_rng(seed).choice(len(points), limit,
                                                           replace=False)]
    return points


# The toy scene for block 1: simple shapes on a ground plane, arranged so that no two
# viewpoints look alike. Flat colours, so each shape matches its dot in the maps.
SCENE_PIECES = [
    ('models/box', (-3.0, 0.0, 2.5), 1.6, (0.85, 0.25, 0.25, 1)),
    ('models/box', (3.5, 0.0, -1.0), 1.0, (0.95, 0.75, 0.20, 1)),
    ('models/teapot', (0.0, 0.0, -3.5), 1.1, (0.25, 0.55, 0.85, 1)),
    ('models/smiley', (-2.0, -1.2, -2.0), 0.9, (0.35, 0.75, 0.35, 1)),
    ('models/frowney', (2.5, -0.6, 3.0), 0.8, (0.75, 0.45, 0.85, 1)),
    ('models/jack', (0.5, -0.4, 0.5), 0.9, (0.95, 0.95, 0.95, 1)),
]
scene_pivots = []


def build_scene():
    ground = renderer.loader.loadModel('models/box')
    ground.setScale(24, 24, 0.2)
    ground.setPos(-12, -12, -0.2)
    grid_texture = renderer.loader.loadTexture('maps/grid.rgb')
    grid_texture.setMinfilter(SamplerState.FT_linear_mipmap_linear)
    grid_texture.setAnisotropicDegree(16)
    ground.setTexture(grid_texture, 1)
    ground.setTexScale(TextureStage.getDefault(), 6, 6)
    ground.reparentTo(renderer.scene_root)
    for model, position_cv, scale, colour in SCENE_PIECES:
        node = renderer.loader.loadModel(model)
        fit_to_unit(node)
        node.setScale(node.getScale() * scale)
        node.setPos(node.getPos() * scale)
        pivot = renderer.scene_root.attachNewNode('piece')
        node.reparentTo(pivot)
        pivot.setPos(cv_to_panda(position_cv))
        pivot.setZ(pivot.getZ() + scale / 2)   # stand it on the ground, not half in it
        pivot.setTextureOff(1)
        pivot.setColor(Vec4(*colour), 1)
        scene_pivots.append(pivot)


build_scene()


def show_all_pieces():
    renderer.scene_root.show()
    renderer.object_root.hide()
    for pivot in scene_pivots:
        pivot.show()


def show_only_cube():
    renderer.scene_root.show()
    renderer.object_root.hide()
    for index, pivot in enumerate(scene_pivots):
        pivot.show() if index == 0 else pivot.hide()


CIRCLE_RADIUS, CIRCLE_HEIGHT = 11.0, -4.5   # Y is down, so this height is above ground


def pose_on_circle(angle_degrees):
    """The one-degree-of-freedom camera: somewhere on a ring, always facing the middle."""
    angle = np.radians(angle_degrees)
    position = (CIRCLE_RADIUS * np.sin(angle), CIRCLE_HEIGHT, CIRCLE_RADIUS * np.cos(angle))
    return look_at(position, (0.0, 0.0, 0.0))


def show(fig):
    """Draw a figure and close it. Without the close, the inline backend still holds the
    figure open at the end of the cell and flushes a second copy of it, outside the Output
    widget -- which is why an exercise would appear twice."""
    display(fig)
    plt.close(fig)


def wire(draw, controls):
    """Draw once now, and again whenever a control moves. Deliberately not
    ipywidgets.interactive_output, which captures into an Output widget of its own that we
    never display -- the drawing then lands nowhere."""
    def redraw(_change=None):
        draw(**{name: widget.value for name, widget in controls.items()})

    for widget in controls.values():
        widget.observe(redraw, names='value')
    redraw()


def dial(description, value, low, high, step=0.5):
    return widgets.FloatSlider(value=value, min=low, max=high, step=step,
                               description=description, continuous_update=False,
                               layout=widgets.Layout(width='330px'))


# --------------------------------------------------------------- the pandas bit
PANDA_FOCAL = 520.0
PANDA_FRONT_VIEW = look_at((0.0, -0.75, -3.1), (0.0, -0.22, 0.0))
# Almost at ground level, so it is obvious whether the feet touch.
PANDA_SIDE_VIEW = look_at((3.0, -0.35, -0.6), (0.0, -0.22, 0.0))
FAMILY_VIEW = look_at((0.0, -1.1, -4.6), (0.0, -0.25, 0.0))

# The panda "arrives" at an awkward size and position, so that centring and normalising it
# is worth doing.
PANDA_ARRIVED_SIZE = 2.6
PANDA_ARRIVED_AT = np.array([2.2, -1.9, 1.5])

panda_template = None
panda_cloud = None
panda_instances = []
ground_plane = None
panda_half_height = 0.5


def show_panda_scene():
    """Block 1's shapes away, the pandas out."""
    renderer.scene_root.hide()
    renderer.object_root.show()


def setup_panda_scene():
    """Loads the panda and a ground plane, and puts block 1's scene away."""
    global panda_template, panda_cloud, ground_plane, panda_half_height
    show_panda_scene()
    if panda_template is None:
        panda_template = renderer.loader.loadModel('models/panda-model')
        fit_to_unit(panda_template)
        fitted = panda_template.getTightBounds()
        panda_half_height = float((fitted[1].z - fitted[0].z) / 2)
        panda_cloud = model_points(panda_template)
        panda_template.detachNode()
    if ground_plane is None:
        ground_plane = renderer.loader.loadModel('models/box')
        ground_plane.setScale(5, 5, 0.02)
        ground_plane.setPos(-2.5, -2.5, 0)
        plane_texture = renderer.loader.loadTexture('maps/grid.rgb')
        plane_texture.setMinfilter(SamplerState.FT_linear_mipmap_linear)
        plane_texture.setAnisotropicDegree(16)
        ground_plane.setTexture(plane_texture, 1)
        ground_plane.setTexScale(TextureStage.getDefault(), 4, 4)
        ground_plane.setColor(Vec4(0.75, 0.85, 0.72, 1), 1)
        ground_plane.reparentTo(renderer.object_root)


def panda_corners():
    """The panda's own surface, as it arrived: off to one side and 2.6 units tall."""
    return panda_cloud * PANDA_ARRIVED_SIZE + PANDA_ARRIVED_AT


def set_ground_height(height_cv):
    ground_plane.setZ(-float(height_cv))


def panda_feet_height(scale=1.0, y=0.0):
    """Where the bottom of a panda sits, in our Y-is-down world."""
    return y + scale * panda_half_height


def place_pandas(specs):
    """One panda per entry; extras get hidden rather than destroyed."""
    while len(panda_instances) < len(specs):
        pivot = renderer.object_root.attachNewNode(f'panda{len(panda_instances)}')
        panda_template.instanceTo(pivot)
        panda_instances.append(pivot)
    for pivot, spec in zip(panda_instances, specs):
        pivot.show()
        pivot.setScale(spec['scale'])
        pivot.setPos(cv_to_panda((spec['x'], spec['y'], spec['z'])))
        pivot.setH(spec['turn'])
    for pivot in panda_instances[len(specs):]:
        pivot.hide()


# ------------------------------------------------------------- the drawing bit
def to_display(v):
    """Our coordinates are X-right, Y-down, Z-forward. A 3D plot reads far better with up
    actually pointing up, so we lay them out as (X, Z, -Y)."""
    v = np.atleast_2d(np.asarray(v, float))
    return np.stack([v[:, 0], v[:, 2], -v[:, 1]], axis=1)


from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d.proj3d import proj_transform


class Arrow3D(FancyArrowPatch):
    """A real arrow between two points in a 3D plot. Matplotlib's own quiver sizes its
    head in data units, which goes wrong as soon as the axes are not cubes."""

    def __init__(self, start, end, **kwargs):
        super().__init__((0, 0), (0, 0), **kwargs)
        self._ends = np.array([start, end], dtype=float)

    def do_3d_projection(self, renderer=None):
        flat_x, flat_y, flat_z = proj_transform(*self._ends.T, self.axes.M)
        self.set_positions((flat_x[0], flat_y[0]), (flat_x[1], flat_y[1]))
        return float(np.min(flat_z))


def draw_arrow(ax, start, end, color, lw=3, head=16):
    """start and end are already in display coordinates."""
    ax.add_artist(Arrow3D(start, end, arrowstyle='-|>', mutation_scale=head, lw=lw,
                          color=color, shrinkA=0, shrinkB=0))


def label_axes(ax):
    ax.set_xlabel('X (right)', fontsize=8)
    ax.set_ylabel('Z (forward)', fontsize=8)
    ax.set_zlabel('height', fontsize=8)
    ax.tick_params(labelsize=6)


def draw_camera(ax, pose, length, colors=('tab:red', 'tab:green', 'tab:blue'),
                marker_color='k', label=None, lw=1.2, alpha=1.0):
    """A camera drawn as its own three axes: right, down, forward."""
    position = to_display(pose['position'])[0]
    for row, colour in zip(rotation_of(pose), colors):
        ax.quiver(*position, *to_display(row)[0], length=length, color=colour, lw=lw,
                  alpha=alpha, arrow_length_ratio=0.25)
    ax.scatter(*position, color=marker_color, s=40, label=label, depthshade=False)


def draw_scene_points(ax, size=55):
    for _, position_cv, _, colour in SCENE_PIECES:
        ax.scatter(*to_display(position_cv)[0], color=colour[:3], s=size, depthshade=False)


def frame_around(ax, points_cv, margin=1.6):
    """Frames a 3D view on whatever it has to show, with equal scales on every axis and the
    height axis starting just under the floor."""
    shown = to_display(np.atleast_2d(points_cv))
    centre = (shown.min(0) + shown.max(0)) / 2
    radius = max((shown.max(0) - shown.min(0)).max() / 2, 1e-6) + margin
    for setter, value in zip((ax.set_xlim, ax.set_ylim), centre):
        setter(value - radius, value + radius)
    top = max(shown[:, 2].max() + margin, 0.5 * radius)
    ax.set_zlim(-margin, top)
    ax.set_box_aspect((2 * radius, 2 * radius, top + margin))


def ground_grid(ax, extent, step=4.0):
    """A faint floor, so heights are readable."""
    for value in np.arange(-extent, extent + step, step):
        ax.plot([-extent, extent], [value, value], [0, 0], color='0.85', lw=0.5)
        ax.plot([value, value], [-extent, extent], [0, 0], color='0.85', lw=0.5)


def draw_box(ax, low, high, color='tab:orange', lw=1.0):
    """The wireframe of a box, from its two opposite corners."""
    corners = np.array([[x, y, z] for x in (low[0], high[0])
                        for y in (low[1], high[1]) for z in (low[2], high[2])])
    shown = to_display(corners)
    for i, a in enumerate(shown):
        for j, b in enumerate(shown):
            if i < j and np.sum(np.abs(corners[i] - corners[j]) > 1e-9) == 1:
                ax.plot(*np.stack([a, b]).T, color=color, lw=lw)


# --------------------------------------------------------- the real reconstruction
BOULDER = Path(DATA) / 'precomputed' / 'boulder'
boulder_meta = json.loads((BOULDER / 'meta.json').read_text())
boulder_track = json.loads((BOULDER / 'camera_track.json').read_text())
boulder_points = np.load(BOULDER / 'points.npy')
boulder_colors = np.load(BOULDER / 'point_colors.npy') / 255.0
camera_ring = np.array([entry['position'] for entry in boulder_track])

# We work at the renderer's size rather than the video's; the focal length scales with it.
BOULDER_FOCAL = boulder_meta['focal_px'] * VIEW_W / boulder_meta['width']


def clip_frame(name, index):
    """One frame of one of the workshop clips, at the size we work at."""
    capture = cv2.VideoCapture(str(Path(DATA) / name))
    capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ok, frame = capture.read()
    capture.release()
    if not ok:
        raise RuntimeError(f'could not read frame {index} of {name}')
    return cv2.cvtColor(cv2.resize(frame, (VIEW_W, VIEW_H)), cv2.COLOR_BGR2RGB)


def boulder_frame(index):
    return clip_frame('boulder.mp4', index)


def boulder_pose(index):
    """The camera pose the reconstruction recovered for that frame."""
    entry = boulder_track[int(index)]
    return dict(position=np.asarray(entry['position'], float),
                forward=np.asarray(entry['forward'], float),
                up=np.asarray(entry['up'], float))


print('Ready.')

# Block 0 — Vectors, quickly

A **vector** is just a few numbers kept together.

Start with two. On a flat map, two numbers are enough to say where something is: how far
**right**, and how far **up**.

```
(2, 1)     ->   2 to the right, 1 up
```

The spot where both are zero is called the **origin**. Everything is measured from there.

In [ ]:
# @title Showcase — a vector on a flat map
# @markdown Drag the two numbers and watch the arrow. The **length** of a vector is simply
# @markdown how far the arrow reaches.

LIMIT = 2.0

flat_x = dial('x  (right)', 1.5, -LIMIT, LIMIT, step=0.25)
flat_y = dial('y  (up)', 1.0, -LIMIT, LIMIT, step=0.25)
out_flat = widgets.Output()


def draw_flat(x, y):
    with out_flat:
        out_flat.clear_output(wait=True)
        fig, panel = plt.subplots(figsize=(4.6, 4.6))
        panel.axhline(0, color='0.75', lw=1)
        panel.axvline(0, color='0.75', lw=1)
        panel.set_xticks(np.arange(-LIMIT, LIMIT + 1, 1))
        panel.set_yticks(np.arange(-LIMIT, LIMIT + 1, 1))
        panel.grid(color='0.9', lw=0.6)

        if abs(x) > 1e-9 or abs(y) > 1e-9:
            panel.annotate('', xy=(x, y), xytext=(0, 0),
                           arrowprops=dict(arrowstyle='-|>', color='tab:blue', lw=2.5))
            panel.plot([0, x], [0, 0], color='0.6', ls=':', lw=1.2)
            panel.plot([x, x], [0, y], color='0.6', ls=':', lw=1.2)
            panel.scatter([x], [y], color='tab:blue', s=45, zorder=3)
            panel.annotate(f'({x:g}, {y:g})', (x, y), textcoords='offset points',
                           xytext=(8, 8), color='tab:blue', fontsize=11)
        panel.scatter([0], [0], color='k', s=40, zorder=3)

        panel.set_xlim(-LIMIT, LIMIT)
        panel.set_ylim(-LIMIT, LIMIT)
        panel.set_aspect('equal')
        panel.set_xlabel('x  (right)')
        panel.set_ylabel('y  (up)')
        fig.tight_layout()
        show(fig)

        print(f'length = square root of ({x:g}x{x:g} + {y:g}x{y:g}) '
              f'= {np.hypot(x, y):.2f}')


display(widgets.VBox([widgets.HBox([flat_x, flat_y]), out_flat]))
wire(draw_flat, dict(x=flat_x, y=flat_y))

## Now the third number

A room is not flat, so two numbers are not enough. Add a third for how far **forward**:

| | |
|---|---|
| **x** | how far right |
| **y** | how far up |
| **z** | how far forward, away from you |

Those three numbers can describe **a place** (where the camera is) or **a direction**
(where it is looking).

In [ ]:
# @title Showcase — the same thing, in 3D
# @markdown A third slider, and a third dashed line. Everything else is the same idea.

space_x = dial('x  (right)', 1.5, -LIMIT, LIMIT, step=0.25)
space_y = dial('y  (up)', 1.0, -LIMIT, LIMIT, step=0.25)
space_z = dial('z  (forward)', 1.0, -LIMIT, LIMIT, step=0.25)
show_unit = widgets.Checkbox(value=False, description='also draw it with length 1')
out_space = widgets.Output()


def draw_space(x, y, z, unit):
    with out_space:
        out_space.clear_output(wait=True)
        v = np.array([x, y, z])
        length = np.linalg.norm(v)

        fig = plt.figure(figsize=(6.0, 5.0))
        panel = fig.add_subplot(1, 1, 1, projection='3d')
        panel.scatter(0, 0, 0, color='k', s=45, label='the origin')
        if length > 1e-9:
            tip = np.array([x, z, y])      # the plot's axes are right, forward, up
            draw_arrow(panel, [0, 0, 0], tip, 'tab:blue')
            panel.plot([], [], color='tab:blue', lw=3, label=f'({x:g}, {y:g}, {z:g})')
            panel.plot([0, tip[0]], [0, 0], [0, 0], color='0.6', ls=':', lw=1.2)
            panel.plot([tip[0], tip[0]], [0, tip[1]], [0, 0], color='0.6', ls=':', lw=1.2)
            panel.plot([tip[0], tip[0]], [tip[1], tip[1]], [0, tip[2]], color='0.6',
                       ls=':', lw=1.2)
            if unit:
                along = v / length
                short = np.array([along[0], along[2], along[1]])
                draw_arrow(panel, [0, 0, 0], short, 'tab:orange')
                panel.plot([], [], color='tab:orange', lw=3,
                           label='same direction, length 1')

        # the box never moves, so the arrow is the only thing that changes
        panel.set_xlim(-LIMIT, LIMIT)
        panel.set_ylim(-LIMIT, LIMIT)
        panel.set_zlim(-LIMIT, LIMIT)
        panel.set_box_aspect((1, 1, 1))
        panel.set_xticks([-2, -1, 0, 1, 2])
        panel.set_yticks([-2, -1, 0, 1, 2])
        panel.set_zticks([-2, -1, 0, 1, 2])
        panel.set_xlabel('x  (right)', fontsize=9)
        panel.set_ylabel('z  (forward)', fontsize=9)
        panel.set_zlabel('y  (up)', fontsize=9)
        panel.tick_params(labelsize=7)
        panel.view_init(elev=22, azim=-55)
        panel.legend(fontsize=8, loc='upper left')
        fig.tight_layout()
        show(fig)

        print(f'length = square root of ({x:g}x{x:g} + {y:g}x{y:g} + {z:g}x{z:g}) '
              f'= {length:.2f}')
        if unit and length > 1e-9:
            small = v / length
            print(f'every number divided by {length:.2f} -> '
                  f'({small[0]:.2f}, {small[1]:.2f}, {small[2]:.2f})')


display(widgets.VBox([widgets.HBox([space_x, space_y, space_z]), show_unit, out_space]))
wire(draw_space,
                           dict(x=space_x, y=space_y, z=space_z, unit=show_unit))

> ## Concept — normalizing
>
> Divide a vector by its own length and you get **length 1**, pointing the same way.
>
> We do this whenever a vector means "that way", because then its length means nothing.

## Adding

**Adding** puts one arrow on the end of the other: walk `a`, then walk `b` from wherever
you ended up. The arithmetic happens to each number separately.

In [ ]:
# @title Showcase — adding two arrows
# @markdown Drag the two arrows around. The dashed one is `b`, moved onto the end of `a`.

ax_dial = dial('a: x', 1.25, -LIMIT, LIMIT, step=0.25)
ay_dial = dial('a: y', 0.5, -LIMIT, LIMIT, step=0.25)
bx_dial = dial('b: x', -0.5, -LIMIT, LIMIT, step=0.25)
by_dial = dial('b: y', 1.25, -LIMIT, LIMIT, step=0.25)
out_sum = widgets.Output()


def draw_sum(ax_, ay, bx, by):
    with out_sum:
        out_sum.clear_output(wait=True)
        a = np.array([ax_, ay])
        b = np.array([bx, by])
        result = a + b

        fig, panel = plt.subplots(figsize=(4.8, 4.8))
        panel.axhline(0, color='0.75', lw=1)
        panel.axvline(0, color='0.75', lw=1)
        panel.grid(color='0.92', lw=0.6)

        def arrow(start, end, colour, width=2.5, style='-|>', dashed=False):
            if np.allclose(start, end):
                return
            panel.annotate('', xy=end, xytext=start,
                           arrowprops=dict(arrowstyle=style, color=colour, lw=width,
                                           linestyle=':' if dashed else '-'))

        arrow((0, 0), a, 'tab:blue')
        panel.annotate('a', a, textcoords='offset points', xytext=(8, 6),
                       color='tab:blue', fontsize=12)
        arrow((0, 0), b, 'tab:green')
        panel.annotate('b', b, textcoords='offset points', xytext=(8, 6),
                       color='tab:green', fontsize=12)
        arrow(a, a + b, 'tab:green', width=2, dashed=True)        # b, moved onto a's tip
        arrow((0, 0), result, 'tab:orange', width=3)
        panel.annotate('a + b', result, textcoords='offset points', xytext=(10, -14),
                       color='tab:orange', fontsize=12)

        panel.scatter([0], [0], color='k', s=35, zorder=3)
        panel.set_xlim(-2 * LIMIT, 2 * LIMIT)
        panel.set_ylim(-2 * LIMIT, 2 * LIMIT)
        panel.set_aspect('equal')
        panel.set_xlabel('x  (right)')
        panel.set_ylabel('y  (up)')
        fig.tight_layout()
        show(fig)

        print(f'({a[0]:g}, {a[1]:g})  +  ({b[0]:g}, {b[1]:g})  '
              f'=  ({result[0]:g}, {result[1]:g})')


display(widgets.VBox([widgets.HBox([ax_dial, ay_dial]),
                      widgets.HBox([bx_dial, by_dial]), out_sum]))
wire(draw_sum, dict(ax_=ax_dial, ay=ay_dial, bx=bx_dial, by=by_dial))

> ## Concept — the two moves
>
> **add** — one arrow after the other. A place plus a direction is a new place.
>
> **times a number** — same direction, longer or shorter. Times `-1` turns it around.

# Block 1 — Where is the camera?

A photo does not only depend on what is in the room. It depends on **where the camera
was standing** and **which way it was pointing**. Change either and you get a different
picture of the same scene.

Below is a little world with some shapes in it. Your job is to work out where the
photographer was standing.

In [ ]:
# @title Exercise 1 — one dial
# @markdown The camera is somewhere on the ring, at a fixed height, always aimed at the
# @markdown middle of the scene. That leaves exactly **one** thing you can change: how
# @markdown far around the ring it is. Turn the dial until your picture matches the
# @markdown target.

TARGET_ANGLE_1 = 117.0

target_pose_1 = pose_on_circle(TARGET_ANGLE_1)
target_image_1 = renderer.render_from(target_pose_1)

angle_slider = widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=1.0,
                                   description='angle', continuous_update=False,
                                   layout=widgets.Layout(width='660px'))
reveal_1 = widgets.Checkbox(value=False, description='show me the answer')
out_1 = widgets.Output()


def draw_exercise_1(angle, reveal):
    with out_1:
        out_1.clear_output(wait=True)
        show_all_pieces()
        pose = pose_on_circle(angle)
        fig = plt.figure(figsize=(12, 3.4))
        panel = fig.add_subplot(1, 3, 1)
        panel.imshow(target_image_1)
        panel.set_title('the target photo', fontsize=10)
        panel.set_axis_off()
        panel = fig.add_subplot(1, 3, 2)
        panel.imshow(renderer.render_from(pose))
        panel.set_title('your photo', fontsize=10)
        panel.set_axis_off()

        plan = fig.add_subplot(1, 3, 3)
        ring = np.radians(np.linspace(0, 360, 200))
        plan.plot(CIRCLE_RADIUS * np.sin(ring), CIRCLE_RADIUS * np.cos(ring), color='0.8')
        for _, position_cv, _, colour in SCENE_PIECES:
            plan.scatter(position_cv[0], position_cv[2], color=colour[:3], s=45)
        plan.scatter(pose['position'][0], pose['position'][2], color='magenta', s=90,
                     zorder=3, label='you')
        if reveal:
            plan.scatter(target_pose_1['position'][0], target_pose_1['position'][2],
                         facecolors='none', edgecolors='k', s=140, lw=2, label='target')
        plan.set_aspect('equal')
        plan.set_title('looking down from above', fontsize=10)
        plan.legend(fontsize=8, loc='upper right')
        plan.set_xticks([])
        plan.set_yticks([])
        fig.tight_layout()
        show(fig)

        if reveal:
            off_by = abs((angle - TARGET_ANGLE_1 + 180) % 360 - 180)
            print(f'target angle {TARGET_ANGLE_1:.0f}, yours {angle:.0f} '
                  f'-- off by {off_by:.0f} degrees')


display(widgets.VBox([widgets.HBox([angle_slider, reveal_1]), out_1]))
wire(draw_exercise_1, dict(angle=angle_slider, reveal=reveal_1))

> ## Concept — degrees of freedom
>
> The **degrees of freedom** of something are how many numbers you are free to choose. A
> train has 1: how far along the track. A boat on a lake has 3: two for where, one for
> which way it faces.
>
> The camera you just moved had 1 — we had already fixed its height, its distance, and the
> fact that it pointed at the middle.

## Letting the camera go

Now nothing is fixed. Three dials move the camera. Three more turn it: left and right,
up and down, and tilting sideways like leaning your head.

That is **6 degrees of freedom**, and it is everything there is to know about where a
camera is. Moving the camera no longer changes where it points — you have to aim it
yourself, and if you wander too far the scene will leave the frame entirely.

In [ ]:
# @title Exercise 2 — six dials
# @markdown Match the target photo again. The 3D view shows how close your camera really
# @markdown is to the one that took it.

# High up and off to one side, looking down at the middle of the scene.
TARGET_PLACE_2 = (8.0, -7.0, -9.0)
TARGET_POSE_2 = pose_from_angles(TARGET_PLACE_2, *angles_towards(TARGET_PLACE_2))
target_image_2 = renderer.render_from(TARGET_POSE_2)




cam_x = dial('x', 0.0, -16, 16)
cam_y = dial('height', -4.0, -12, -0.5)
cam_z = dial('z', -13.0, -16, 16)
cam_turn = dial('turn', 0.0, -180, 180, step=2.0)
cam_tilt = dial('tilt (+ is up)', 0.0, -60, 60, step=2.0)
cam_roll = dial('roll', 0.0, -45, 45, step=2.0)
reveal_2 = widgets.Checkbox(value=False, description='show me the answer')
out_2 = widgets.Output()


def draw_six_dials(x, y, z, turn, tilt, roll, reveal, target_pose, target_image,
                   output, single_cube=False):
    with output:
        output.clear_output(wait=True)
        show_only_cube() if single_cube else show_all_pieces()
        pose = pose_from_angles((x, y, z), turn, tilt, roll)
        fig = plt.figure(figsize=(12.5, 3.6))
        panel = fig.add_subplot(1, 3, 1)
        panel.imshow(target_image)
        panel.set_title('the target photo', fontsize=10)
        panel.set_axis_off()
        panel = fig.add_subplot(1, 3, 2)
        panel.imshow(renderer.render_from(pose))
        panel.set_title('your photo', fontsize=10)
        panel.set_axis_off()

        space = fig.add_subplot(1, 3, 3, projection='3d')
        pieces = [SCENE_PIECES[0]] if single_cube else SCENE_PIECES
        for _, position_cv, _, colour in pieces:
            space.scatter(*to_display(position_cv)[0], color=colour[:3], s=55,
                          depthshade=False)
        draw_camera(space, pose, 3.0, marker_color='magenta', label='your camera')
        if reveal:
            draw_camera(space, target_pose, 3.0, marker_color='k',
                        label='camera that took it', alpha=0.4)
            space.plot(*to_display([pose['position'], target_pose['position']]).T,
                       color='magenta', ls=':', lw=1.4)
        ground_grid(space, 10)
        frame_around(space, [piece[1] for piece in pieces] +
                     [pose['position'], target_pose['position']])
        space.view_init(elev=26, azim=-62)
        label_axes(space)
        space.legend(fontsize=7, loc='upper left')
        fig.tight_layout()
        show(fig)

        if reveal:
            gap, degrees = pose_error(pose, target_pose)
            print(f'your camera is {gap:.1f} units from the real one, '
                  f'and aimed {degrees:.0f} degrees off')


display(widgets.VBox([widgets.HBox([cam_x, cam_y, cam_z]),
                      widgets.HBox([cam_turn, cam_tilt, cam_roll]),
                      reveal_2, out_2]))
wire(lambda x, y, z, turn, tilt, roll, reveal: draw_six_dials(
        x, y, z, turn, tilt, roll, reveal, TARGET_POSE_2, target_image_2, out_2),
    dict(x=cam_x, y=cam_y, z=cam_z, turn=cam_turn, tilt=cam_tilt, roll=cam_roll,
         reveal=reveal_2))

### Quick question

Think about walking around in Minecraft. Your head is the camera.

**How many degrees of freedom does it have?** Count them: what can you change about where
you are and where you are looking — and what can you *not* change?

## Harder: one cube, no landmarks

The last round was made easy by the scene. Six differently coloured shapes tell you
immediately which side you are on.

Now there is one cube. Same six dials.

In [ ]:
# @title Exercise 3 — six dials, one cube
# @markdown With nothing to compare against, many different camera positions produce
# @markdown almost the same picture. Get as close as you can, then reveal the answer and
# @markdown see how far off you actually were.

TARGET_PLACE_3 = (-6.0, -5.0, -7.0)
TARGET_POSE_3 = pose_from_angles(TARGET_PLACE_3, *angles_towards(TARGET_PLACE_3))
show_only_cube()
target_image_3 = renderer.render_from(TARGET_POSE_3)
show_all_pieces()

cube_x = dial('x', 0.0, -16, 16)
cube_y = dial('height', -4.0, -12, -0.5)
cube_z = dial('z', -13.0, -16, 16)
cube_turn = dial('turn', 0.0, -180, 180, step=2.0)
cube_tilt = dial('tilt (+ is up)', 0.0, -60, 60, step=2.0)
cube_roll = dial('roll', 0.0, -45, 45, step=2.0)
reveal_3 = widgets.Checkbox(value=False, description='show me the answer')
out_3 = widgets.Output()

display(widgets.VBox([widgets.HBox([cube_x, cube_y, cube_z]),
                      widgets.HBox([cube_turn, cube_tilt, cube_roll]),
                      reveal_3, out_3]))
wire(lambda x, y, z, turn, tilt, roll, reveal: draw_six_dials(
        x, y, z, turn, tilt, roll, reveal, TARGET_POSE_3, target_image_3, out_3,
        single_cube=True),
    dict(x=cube_x, y=cube_y, z=cube_z, turn=cube_turn, tilt=cube_tilt, roll=cube_roll,
         reveal=reveal_3))

## How do you write a camera down?

Suppose you had to store a camera in a file, so a computer could reproduce your photo
exactly. You would obviously need **where it stands** — three numbers, a point in space.
And you would need **which way it looks** — an arrow.

Here they are, drawn in the world.

In [ ]:
# @title A position and a direction
# @markdown Everything a camera needs? Have a look, and decide before you scroll on.

SHOWCASE_PLACE = np.array([6.0, -4.0, -9.0])
SHOWCASE_FORWARD = normalize(np.array([0.0, 0.0, 0.0]) - SHOWCASE_PLACE)

fig = plt.figure(figsize=(6.5, 4.8))
space = fig.add_subplot(1, 1, 1, projection='3d')
draw_scene_points(space, size=45)
ground_grid(space, 8)
place = to_display(SHOWCASE_PLACE)[0]
space.scatter(*place, color='k', s=80, label='where it stands (3 numbers)')
space.quiver(*place, *to_display(SHOWCASE_FORWARD)[0], length=8.0, color='tab:blue',
             lw=3.0, arrow_length_ratio=0.15, label='which way it looks (an arrow)')
frame_around(space, [piece[1] for piece in SCENE_PIECES] + [SHOWCASE_PLACE], margin=2.0)
# Looked at side-on, so the arrow does not point away from you and vanish.
space.view_init(elev=20, azim=34)
label_axes(space)
space.legend(fontsize=8, loc='upper left')
fig.tight_layout()
show(fig)

### Is that enough?

If you know exactly where a camera stands, and exactly which way it points, is the photo
settled — or could two cameras agree on both and still take different pictures?

Decide before you run the next cell.

<br><br><br><br><br><br><br><br>

In [ ]:
# @title Showcase — what is still free?
# @markdown Every picture below is taken from the same spot, pointing exactly the same
# @markdown way. The dial changes nothing about where the camera is or what it looks at.

roll_dial = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=5.0,
                                description='the free one', continuous_update=False,
                                layout=widgets.Layout(width='420px'))
out_4 = widgets.Output()


def draw_showcase(roll_degrees):
    with out_4:
        out_4.clear_output(wait=True)
        show_all_pieces()
        forward = SHOWCASE_FORWARD
        angle = np.radians(roll_degrees)
        start_up = camera_axes(forward, WORLD_UP)[1]
        up = (start_up * np.cos(angle) + np.cross(forward, start_up) * np.sin(angle) +
              forward * np.dot(forward, start_up) * (1 - np.cos(angle)))
        pose = dict(position=SHOWCASE_PLACE, forward=forward, up=normalize(up))

        fig = plt.figure(figsize=(11, 3.6))
        panel = fig.add_subplot(1, 2, 1)
        panel.imshow(renderer.render_from(pose))
        panel.set_title('what the camera sees', fontsize=10)
        panel.set_axis_off()

        space = fig.add_subplot(1, 2, 2, projection='3d')
        place = to_display(pose['position'])[0]
        for arrow, colour, name in ((forward, 'tab:blue', 'forward (unchanged)'),
                                    (pose['up'], 'tab:green', 'up (the free one)')):
            space.quiver(*place, *to_display(arrow)[0], length=4.0, color=colour, lw=1.8,
                         arrow_length_ratio=0.22, label=name)
        draw_scene_points(space, size=35)
        ground_grid(space, 8)
        frame_around(space, [piece[1] for piece in SCENE_PIECES] + [pose['position']],
                     margin=2.5)
        space.view_init(elev=20, azim=34)
        label_axes(space)
        space.legend(fontsize=7, loc='upper left')
        fig.tight_layout()
        show(fig)

        print(f'position {np.round(pose["position"], 2)}   '
              f'forward {np.round(forward, 3)}   -- identical every time')


display(widgets.VBox([roll_dial, out_4]))
wire(draw_showcase, dict(roll_degrees=roll_dial))

So a position and a direction are **not** enough: the camera can still spin about its own
line of sight, and the photo turns with it. Aiming takes **3 numbers** — two to fix a
direction, one more for that spin. Exactly the three dials from Exercise 2.

We pin the spin down with a second arrow, **`up`**, which is why tomorrow's file gives every
frame a `position`, a `forward` and an `up`. Nobody stores `right` — one vector operation
works it out from the other two, and **tomorrow you write that**, in `compute_camera_basis`.

The same 3 numbers also get written as **heading, pitch and roll** (tomorrow's `hpr`), a
**rotation matrix**, or a **quaternion**.

> ## Concept — a camera pose
>
> A **pose** is where a camera is and which way it faces: **6 numbers**, three for the
> position and three for the aim.
>
> Tomorrow's notebook writes each one down as `position`, `forward` and `up`, one per
> frame of your video. A whole list of them is called a **camera track** — the path the
> camera took through the world.

## How big is it?

One last thing about cameras, and it is an uncomfortable one: **a photo cannot tell you how
big anything is.**

### Before you run the next cell — a prediction

The next exercise has one dial. Turning it up multiplies **everything in the world** by
that number:

1. the size of every shape
2. the distance between the shapes
3. the height of the camera above the ground
4. the distance from the camera to the shapes

So at 3x, every shape is three times bigger, three times further apart, and the camera
is three times further away.

**What do you think the photo will look like?** Decide before you drag anything.

In [ ]:
# @title Exercise 4 — multiply the whole world
# @markdown Drag it slowly, and keep an eye on the photo rather than the map.

size_dial = widgets.FloatSlider(value=1.0, min=0.4, max=5.0, step=0.2,
                                description='world size', continuous_update=False,
                                layout=widgets.Layout(width='520px'))
out_5 = widgets.Output()
BASE_POSE_5 = look_at((6.0, -5.0, -11.0), (0.0, 0.0, 0.0))


def draw_exercise_5(size):
    with out_5:
        out_5.clear_output(wait=True)
        show_all_pieces()
        renderer.scene_root.setScale(float(size))
        pose = dict(position=np.asarray(BASE_POSE_5['position']) * size,
                    forward=BASE_POSE_5['forward'], up=BASE_POSE_5['up'])
        image = renderer.render_from(pose)
        renderer.scene_root.setScale(1.0)

        fig = plt.figure(figsize=(11, 3.4))
        panel = fig.add_subplot(1, 2, 1)
        panel.imshow(image)
        panel.set_axis_off()
        panel.set_title(f'the photo, with the world {size:g}x its original size', fontsize=10)

        plan = fig.add_subplot(1, 2, 2)
        for _, position_cv, scale, colour in SCENE_PIECES:
            plan.add_patch(plt.Circle((position_cv[0], position_cv[2]), scale / 2,
                                      color='0.85'))
            plan.add_patch(plt.Circle((position_cv[0] * size, position_cv[2] * size),
                                      scale * size / 2, color=colour[:3]))
        plan.scatter(*np.asarray(BASE_POSE_5['position'])[[0, 2]], color='0.7', s=40)
        plan.scatter(pose['position'][0], pose['position'][2], color='magenta', s=70,
                     label='camera', zorder=3)
        plan.plot([pose['position'][0], 0], [pose['position'][2], 0], color='magenta',
                  ls=':', lw=1)
        plan.set_xlim(-70, 70)
        plan.set_ylim(-70, 70)
        plan.set_aspect('equal')
        plan.set_title('seen from above (grey = the original size)', fontsize=10)
        plan.legend(fontsize=8, loc='upper right')
        plan.set_xlabel(f'the camera is now {np.linalg.norm(pose["position"]):.0f} units away',
                        fontsize=9)
        fig.tight_layout()
        show(fig)


display(widgets.VBox([size_dial, out_5]))
wire(draw_exercise_5, dict(size=size_dial))

The map changes enormously. The photo does not change at all — not by a single pixel.

A pebble photographed from 10 cm away and a cliff photographed from 40 m away can produce
exactly the same picture. Nothing in the pixels can separate them.

### So is size just unknowable?

Not quite. What made the photo stay identical was that **everything grew together**.
Break that, and the picture does change. The next exercise gives you two separate dials
so you can see which is which.

In [ ]:
# @title Exercise 5 — two dials that used to be one
# @markdown One dial resizes the scene. The other moves the camera nearer or further.
# @markdown Change only one at a time. Then try to cancel one out with the other.

scene_dial = widgets.FloatSlider(value=1.0, min=0.4, max=4.0, step=0.2,
                                 description='scene size', continuous_update=False,
                                 layout=widgets.Layout(width='330px'))
distance_dial = widgets.FloatSlider(value=1.0, min=0.4, max=4.0, step=0.2,
                                    description='camera distance', continuous_update=False,
                                    layout=widgets.Layout(width='330px'))
out_6 = widgets.Output()


def draw_exercise_6(scene_size, distance):
    with out_6:
        out_6.clear_output(wait=True)
        show_all_pieces()
        renderer.scene_root.setScale(float(scene_size))
        pose = dict(position=np.asarray(BASE_POSE_5['position']) * distance,
                    forward=BASE_POSE_5['forward'], up=BASE_POSE_5['up'])
        image = renderer.render_from(pose)
        renderer.scene_root.setScale(1.0)

        fig = plt.figure(figsize=(7.5, 4.0))
        panel = fig.add_subplot(1, 1, 1)
        panel.imshow(image)
        panel.set_axis_off()
        matched = 'the same photo as before' if abs(scene_size - distance) < 1e-6 \
            else 'a different photo'
        panel.set_title(f'scene {scene_size:g}x, camera {distance:g}x further away '
                        f'-- {matched}', fontsize=11)
        fig.tight_layout()
        show(fig)


display(widgets.VBox([widgets.HBox([scene_dial, distance_dial]), out_6]))
wire(draw_exercise_6,
                           dict(scene_size=scene_dial, distance=distance_dial))

Only the **ratio** between the two matters. Set them to the same number and you are back to
the photo you started with — so no photo, and no pile of photos, can pin down size on its
own.

Park that. It comes back the moment we have something reconstructed, and we deal with it
then.

# Block 2 — From photos to a reconstruction

Everything so far, you were told where the camera was. Tomorrow nobody tells you. You hand
a computer a video, and it has to work out from the pixels alone both what the scene looks
like in 3D *and* where the camera was for every single frame.

> ## Concept — structure from motion
>
> Recovering the **structure** (the scene, in 3D) and the **motion** (where the camera was,
> frame by frame) from nothing but photos. A simplified pipeline:
>
> 1. **find** distinctive points in each photo
> 2. **match** them between photos
> 3. work out **where the camera was** for each photo
> 4. work out the **3D positions** of the points
> 5. **polish** until it all agrees
>
> Cameras come before points: a point's position is two lines of sight crossing, and you
> cannot draw those lines until you know where the cameras stood. Steps 3 and 4 then take
> turns — seed from one pair, place a photo, add its points, place the next.
>
> Today: steps **1 and 2** by hand, then step **3**.

## The footage

Here is a stretch of a clip of a boulder. The camera works its way around it — these are a
few frames from part of that trip.

In [ ]:
# @title Exercise 6 — a look at the clip
# @markdown First the stretch of footage itself, then six frames from it. Notice how much
# @markdown of the rock two neighbouring frames share, and how quickly that falls away as
# @markdown they get further apart.

from IPython.display import HTML, Video

CLIP_FRAMES = [280, 292, 304, 316, 328, 340]

# Cut out just the stretch those frames come from, so the motion is visible rather than
# only implied.
import tempfile
stretch = Path(tempfile.gettempdir()) / 'boulder_stretch.mp4'
if not stretch.exists():
    fps = boulder_meta['fps']
    subprocess.run(
        ['ffmpeg', '-v', 'error', '-y', '-i', str(Path(DATA) / 'boulder.mp4'),
         '-ss', f'{CLIP_FRAMES[0] / fps:.3f}',
         '-t', f'{(CLIP_FRAMES[-1] - CLIP_FRAMES[0] + 1) / fps:.3f}',
         '-vf', 'scale=480:-2', '-an', str(stretch)], check=True)
display(Video(str(stretch), embed=True,
              html_attributes='controls autoplay loop muted'))

fig, axes = plt.subplots(2, 3, figsize=(13, 4.6))
for ax, index in zip(axes.ravel(), CLIP_FRAMES):
    ax.imshow(boulder_frame(index))
    ax.set_title(f'frame {index}', fontsize=9)
    ax.set_axis_off()
fig.tight_layout()
show(fig)

## Your turn to be the matcher

Take two of those frames — a moment apart, so the rock has turned a little. Ten spots are
marked in the first one.

**Find the same ten spots in the second.** That is step 2 of the pipeline, done by hand.

In [ ]:
# @title Exercise 7 — click the same ten spots
# @markdown Click each numbered spot in the second photo, in order. Take them seriously --
# @markdown what happens later depends on how well you do here.

FRAME_A, FRAME_B = 300, 320
frame_a, frame_b = boulder_frame(FRAME_A), boulder_frame(FRAME_B)
pose_a, pose_b = boulder_pose(FRAME_A), boulder_pose(FRAME_B)


def texture_score(image, xy, radius=9):
    """How much detail surrounds a spot: high on a rough corner, near zero on smooth stone
    or on empty background."""
    x, y = int(round(xy[0])), int(round(xy[1]))
    patch = image[max(y - radius, 0):y + radius, max(x - radius, 0):x + radius]
    if patch.size == 0:
        return 0.0
    return float(cv2.Laplacian(cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())


def points_visible_in_both(margin=70):
    pixels_a, front_a = project(pose_a, boulder_points, BOULDER_FOCAL, VIEW_W, VIEW_H)
    pixels_b, front_b = project(pose_b, boulder_points, BOULDER_FOCAL, VIEW_W, VIEW_H)

    def inside(pixels, front):
        return (front & (pixels[:, 0] > margin) & (pixels[:, 0] < VIEW_W - margin) &
                (pixels[:, 1] > margin) & (pixels[:, 1] < VIEW_H - margin))

    return np.where(inside(pixels_a, front_a) & inside(pixels_b, front_b))[0], pixels_a


def findable_points(count=8, apart=64, tolerance=3.0):
    """Points the matching algorithm itself gets right between these two photos: it finds
    a feature in both, and that feature sits on a reconstructed point. If a computer can
    find them, so can you -- which is what makes this a fair exercise."""
    finder = cv2.SIFT_create(nfeatures=3000)
    pairs = cv2.BFMatcher()
    keys_a, description_a = finder.detectAndCompute(
        cv2.cvtColor(frame_a, cv2.COLOR_RGB2GRAY), None)
    keys_b, description_b = finder.detectAndCompute(
        cv2.cvtColor(frame_b, cv2.COLOR_RGB2GRAY), None)
    matches = [pair[0] for pair in pairs.knnMatch(description_a, description_b, k=2)
               if len(pair) == 2 and pair[0].distance < 0.75 * pair[1].distance]

    candidates, pixels_a = points_visible_in_both()
    pixels_b, _ = project(pose_b, boulder_points, BOULDER_FOCAL, VIEW_W, VIEW_H)

    agreed = []
    for match in matches:
        here = np.array(keys_a[match.queryIdx].pt)
        there = np.array(keys_b[match.trainIdx].pt)
        distances = np.linalg.norm(pixels_a[candidates] - here, axis=1)
        nearest = candidates[int(np.argmin(distances))]
        # the feature must sit on a reconstructed point, and that point must agree about
        # where it ends up in the second photo
        if distances.min() < tolerance and np.linalg.norm(pixels_b[nearest] - there) < tolerance:
            agreed.append(nearest)

    detail = np.array([texture_score(frame_a, pixels_a[index]) for index in agreed])
    chosen = []
    for index in np.array(agreed)[np.argsort(-detail)]:
        if len(chosen) >= count:
            break
        if all(np.linalg.norm(pixels_a[index] - pixels_a[other]) > apart for other in chosen):
            chosen.append(index)
    return chosen


# Nine spots the algorithm itself matches correctly, ordered so the crispest come first and
# the flattest of them lands last, plus one that is not on the rock at all.
on_rock = findable_points(count=9)
marked_points_3d = boulder_points[np.array(on_rock)]
solid_in_a, _ = project(pose_a, marked_points_3d, BOULDER_FOCAL, VIEW_W, VIEW_H)
solid_in_b, _ = project(pose_b, marked_points_3d, BOULDER_FOCAL, VIEW_W, VIEW_H)

BACKGROUND_SPOT = np.array([VIEW_W * 0.13, VIEW_H * 0.22])
marked_in_a = np.vstack([solid_in_a, BACKGROUND_SPOT])
truth_in_b = np.vstack([solid_in_b, [np.nan, np.nan]])   # nothing to find, by design
ON_THE_ROCK = len(on_rock)          # spots 1..9 sit on stone; spot 10 does not

import base64

from IPython.display import HTML


def draw_spots(image, spots, colour=(255, 215, 0)):
    """The numbered rings, drawn into the picture itself so nothing gets resampled."""
    marked_image = image.copy()
    for number, spot in enumerate(spots, start=1):
        centre = (int(round(spot[0])), int(round(spot[1])))
        cv2.circle(marked_image, centre, 9, colour, 2)
        cv2.putText(marked_image, str(number), (centre[0] + 11, centre[1] - 9),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1, cv2.LINE_AA)
    return marked_image


def as_png(image):
    ok, buffer = cv2.imencode('.png', cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    return base64.b64encode(buffer).decode()


photo_1 = draw_spots(frame_a, marked_in_a)

# Photo 1 at its true size, so what you see is what the coordinates mean. Photo 2 is the
# canvas you click on, drawn just below at exactly the same size.
display(HTML(f"""
<div style="font-family: sans-serif">
  <div><b>photo 1 — the ten spots</b></div>
  <img src="data:image/png;base64,{as_png(photo_1)}" width="{VIEW_W}" height="{VIEW_H}">
</div>
"""))

# Clicking happens in the browser. Off Colab we stand in for it, so the rest still runs.
your_clicks = np.where(np.isnan(truth_in_b), marked_in_a, truth_in_b) + \
    np.random.default_rng(1).normal(0, 6, marked_in_a.shape)
try:
    from google.colab.output import eval_js

    def capture_clicks(total):
        """One canvas, drawn at exactly the picture's own size so a click lands on the
        pixel it looks like it lands on. Each click is ringed as you make it."""
        script = """
        async function getClicks() {
          const box = document.createElement('div');
          box.style.fontFamily = 'sans-serif';
          const heading = document.createElement('div');
          heading.innerHTML = '<b>photo 2 — click the same spots here</b>';
          heading.style.margin = '8px 0 4px';
          const hint = document.createElement('div');
          hint.style.fontWeight = 'bold';
          hint.style.margin = '6px 0';
          const readout = document.createElement('div');
          readout.style.font = '12px monospace';
          readout.style.margin = '6px 0';
          readout.style.minHeight = '110px';      // reserved, so the canvas never moves
          const canvas = document.createElement('canvas');
          canvas.width = IMG_W; canvas.height = IMG_H;
          canvas.style.width = IMG_W + 'px';        // no scaling: 1 css pixel = 1 image pixel
          canvas.style.height = IMG_H + 'px';
          canvas.style.cursor = 'crosshair';
          box.appendChild(heading); box.appendChild(hint);
          box.appendChild(canvas); box.appendChild(readout);
          document.body.appendChild(box);

          const img = new Image();
          await new Promise(done => { img.onload = done; img.src = 'data:image/png;base64,IMAGE_DATA'; });
          const ctx = canvas.getContext('2d');
          ctx.drawImage(img, 0, 0);

          const picked = [];
          for (let n = 1; n <= TOTAL; n++) {
            hint.textContent = 'Click spot ' + n + ' of ' + TOTAL;
            const point = await new Promise(done => {
              canvas.onclick = (event) => {
                const rect = canvas.getBoundingClientRect();
                const x = (event.clientX - rect.left) * (canvas.width / rect.width);
                const y = (event.clientY - rect.top) * (canvas.height / rect.height);
                canvas.onclick = null;
                done([x, y]);
              };
            });
            ctx.strokeStyle = '#00e5ff'; ctx.lineWidth = 2;
            ctx.beginPath(); ctx.arc(point[0], point[1], 9, 0, 2 * Math.PI); ctx.stroke();
            ctx.fillStyle = '#00e5ff'; ctx.font = 'bold 13px sans-serif';
            ctx.fillText(String(n), point[0] + 11, point[1] - 9);
            picked.push(point);
            readout.textContent += 'spot ' + n + ': (' +
                Math.round(point[0]) + ', ' + Math.round(point[1]) + ')   ';
          }
          hint.textContent = 'All ' + TOTAL + ' clicked. Your rings are drawn above.';
          return picked;
        }
        getClicks()
        """.replace('IMAGE_DATA', as_png(frame_b)) \
           .replace('IMG_W', str(VIEW_W)).replace('IMG_H', str(VIEW_H)) \
           .replace('TOTAL', str(total))
        return eval_js(script)

    your_clicks = np.array(capture_clicks(len(marked_in_a)), dtype=float)
except ImportError:
    print('(not running on Colab -- using stand-in clicks so the rest still works)')

In [ ]:
# @title Exercise 7 — how did you do?
# @markdown Each arrow runs from where you clicked to where that spot really is. We can
# @markdown only draw those because this rock was reconstructed in advance, so the answers
# @markdown were already known. In a real reconstruction nobody knows them -- which is
# @markdown exactly what makes a wrong match dangerous. Nothing flags it.

errors = np.full(len(marked_in_a), np.nan)
errors[:ON_THE_ROCK] = np.linalg.norm(
    your_clicks[:ON_THE_ROCK] - truth_in_b[:ON_THE_ROCK], axis=1)

# Drawn into the picture at its own size, so it lines up pixel for pixel with the one you
# clicked on.
GOOD, BAD, UNKNOWABLE = (60, 230, 90), (235, 60, 60), (0, 210, 255)
verdict_image = frame_b.copy()
for number, (click, truth) in enumerate(zip(your_clicks, truth_in_b), start=1):
    here = (int(round(click[0])), int(round(click[1])))
    if np.isnan(truth).any():
        cv2.circle(verdict_image, here, 9, UNKNOWABLE, 2)
        cv2.putText(verdict_image, f'{number}?', (here[0] + 11, here[1] - 9),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, UNKNOWABLE, 1, cv2.LINE_AA)
        continue
    colour = GOOD if errors[number - 1] < 12 else BAD
    there = (int(round(truth[0])), int(round(truth[1])))
    cv2.circle(verdict_image, here, 9, colour, 2)
    if np.linalg.norm(np.array(there) - np.array(here)) > 12:
        cv2.arrowedLine(verdict_image, here, there, colour, 2, cv2.LINE_AA, tipLength=0.25)
    else:
        cv2.drawMarker(verdict_image, there, colour, cv2.MARKER_CROSS, 12, 2, cv2.LINE_AA)
    cv2.putText(verdict_image, str(number), (here[0] + 11, here[1] - 9),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1, cv2.LINE_AA)

display(HTML(f"""
<div style="font-family: sans-serif">
  <div><b>your ring, and a line to where the spot really is</b></div>
  <img src="data:image/png;base64,{as_png(verdict_image)}" width="{VIEW_W}" height="{VIEW_H}">
</div>
"""))

scored = errors[:ON_THE_ROCK]
print(f'spots 1-{ON_THE_ROCK}: off by {np.nanmean(scored):.1f} pixels on average '
      f'(best {np.nanmin(scored):.1f}, worst {np.nanmax(scored):.1f})')
print(f'spot {len(marked_in_a)}: no arrow, because there is no right answer for it')

### Talk about it

- **Which spots were easy?** Which did you find instantly, and what did those bits of rock
  look like?
- **Which were hard?** The later ones sit on flatter, plainer rock than the first few.
  What went wrong there, and would a second attempt do any better?
- **Spot 10 has no arrow at all.** It was never on the rock — it is a point in the empty
  black background. What would the *right* answer even be? And what would it mean for a
  computer to be confident about a spot like that?

### Now generalise

You have just done, ten times over, the job a computer does millions of times.

**Where would it do well, and where would it struggle?** Picture real things you could
point a camera at, and sort them into easy and hard: a brick wall, a freshly painted door,
a gravel path, a window, a calm lake, a bookshelf, a green screen.

Commit to your guesses. The next few cells test them.

> ## Concept — feature detection and feature matching
>
> **Feature detection** is step 1: scanning a photo for spots distinctive enough to be
> recognised again — corners, specks, sharp changes. A pixel in the middle of a blank wall
> looks exactly like its neighbours, so there is nothing to recognise it by.
>
> **Feature matching** is step 2: for each of those spots, hunting through the other photo
> for the one that looks most like it.
>
> Both work from a small patch of pixels. The computer has no idea what a rock is, or a
> wall — only patches.

In [ ]:
# @title Showcase — what the algorithm found on the rock
# @markdown Every line joins a spot in the left photo to the spot the algorithm believes is
# @markdown the same place in the right one.

detector = cv2.SIFT_create(nfeatures=1500)
matcher = cv2.BFMatcher()


def find_and_match(image_a, image_b, ratio=0.75):
    """Step 1 and step 2, on a pair of photos."""
    grey_a = cv2.cvtColor(image_a, cv2.COLOR_RGB2GRAY)
    grey_b = cv2.cvtColor(image_b, cv2.COLOR_RGB2GRAY)
    keys_a, description_a = detector.detectAndCompute(grey_a, None)
    keys_b, description_b = detector.detectAndCompute(grey_b, None)
    if description_a is None or description_b is None or len(keys_a) < 2 or len(keys_b) < 2:
        return keys_a, keys_b, []
    good = [pair[0] for pair in matcher.knnMatch(description_a, description_b, k=2)
            if len(pair) == 2 and pair[0].distance < ratio * pair[1].distance]
    return keys_a, keys_b, good


def show_pair(image_a, image_b, title):
    """The two photos, side by side, with nothing drawn on them yet."""
    fig, panel = plt.subplots(figsize=(14, 4))
    panel.imshow(np.hstack([image_a, image_b]))
    panel.set_axis_off()
    panel.set_title(title, fontsize=11)
    fig.tight_layout()
    show(fig)


def matches_with_reveal(image_a, image_b, title):
    """Look first, argue about it, then let the algorithm answer."""
    reveal = widgets.Checkbox(value=False, description='show me the matches')
    output = widgets.Output()

    def draw(show):
        with output:
            output.clear_output(wait=True)
            if show:
                show_matches(image_a, image_b, title)
            else:
                show_pair(image_a, image_b, f'{title} -- how many matches, and where?')

    display(widgets.VBox([reveal, output]))
    wire(draw, dict(show=reveal))


def show_matches(image_a, image_b, title, limit=80):
    keys_a, keys_b, good = find_and_match(image_a, image_b)
    shown = sorted(good, key=lambda match: match.distance)[:limit]
    drawn = cv2.drawMatches(image_a, keys_a, image_b, keys_b, shown, None,
                            matchColor=(80, 220, 120), singlePointColor=(240, 200, 60),
                            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    fig, panel = plt.subplots(figsize=(14, 4))
    panel.imshow(drawn)
    panel.set_axis_off()
    panel.set_title(f'{title}\n{len(keys_a)} and {len(keys_b)} points found, '
                    f'{len(good)} matched (drawing the best {len(shown)})', fontsize=11)
    fig.tight_layout()
    show(fig)


show_matches(frame_a, frame_b, 'the boulder, two frames apart')

## A harder subject

The next clip is a phone panning across a plain white wall. Nothing is wrong with it — in
focus, well lit, camera moving steadily.

**Before you run it:** how many findable spots do you expect, and *where* in the frame?
How many of those do you think will match correctly between two frames?

In [ ]:
# @title Exercise 8 — the blank wall
# @markdown Same detector, same matcher, same settings as on the rock. Look at the pair
# @markdown first and commit to a guess -- tick the box only once you have.

wall_a, wall_b = clip_frame('wall.mp4', 20), clip_frame('wall.mp4', 90)
matches_with_reveal(wall_a, wall_b, 'a blank wall, two frames apart')

## And one more

This clip was shot while whipping the camera around fast. The subject is a Lego model on a
table — plenty of texture, plenty of corners, nothing blank about it at all.

**Before you run it:** the scene is full of detail this time. Does that save it? Where do
you expect matches, and roughly how many?

In [ ]:
# @title Exercise 9 — motion blur
# @markdown Same detector and matcher again. Guess first, tick second.

lego_a, lego_b = clip_frame('lego.mp4', 2), clip_frame('lego.mp4', 10)
matches_with_reveal(lego_a, lego_b, 'a fast, blurry pan, a few frames apart')

Blur smears every corner into a streak, so there is nothing sharp left to recognise. That
is why tomorrow's pipeline measures how sharp each frame is and drops the worst ones before
it even starts matching.

These three runs were prepared in advance, so they come back instantly. **Tomorrow you run
the real thing, live, on your own footage** — which is exactly why it matters that you film
something with texture, and film it slowly.

## Step 3: where was the camera?

A pose is 6 unknowns. Every point you matched gives 2 equations — the across and the down
of where it landed. Nine points give 18 equations for 6 unknowns.

That is a **linear system**, over-determined: no pose satisfies all of it exactly, so we
take the one that comes closest, with the leftover error spread as thinly as it will go.
Small mistakes cancel; big ones drag the answer with them.

A real reconstruction solves one of these for every photo — each newly placed photo
reveals more 3D points, and those points are what place the photo after it. Yours is one.

In [ ]:
# @title Exercise 10 — solve for the camera
# @markdown Left: the whole scene, with the camera your clicks imply. Middle: zoomed right
# @markdown in, so the gap is visible at all. Right: your clicks against where your camera
# @markdown says those points should have landed. A good solve leaves almost nothing to
# @markdown draw. **Every one of your clicks is used**, mistakes included. Spot 10 sits
# @markdown this one out -- it has no 3D position to work from.

CAMERA_MATRIX = np.array([[BOULDER_FOCAL, 0, VIEW_W / 2],
                          [0, BOULDER_FOCAL, VIEW_H / 2],
                          [0, 0, 1]])


def solve_for_camera(points_3d, clicks):
    """Where must the camera have been? Every point you hand it counts -- nothing is
    quietly thrown away. Dropping the bad ones is your job, in the next cell."""
    points_3d = np.asarray(points_3d, np.float64)
    clicks = np.asarray(clicks, np.float64)
    method = getattr(cv2, 'SOLVEPNP_SQPNP', cv2.SOLVEPNP_ITERATIVE)
    ok, rotation_vector, translation = cv2.solvePnP(points_3d, clicks, CAMERA_MATRIX,
                                                    None, flags=method)
    if not ok:
        return None
    rotation_vector, translation = cv2.solvePnPRefineLM(
        points_3d, clicks, CAMERA_MATRIX, None, rotation_vector, translation)
    R, _ = cv2.Rodrigues(rotation_vector)
    return dict(position=(-R.T @ translation).ravel(), forward=R[2], up=-R[1])


def show_solution(points_3d, clicks, title):
    solved = solve_for_camera(points_3d, clicks)
    if solved is None:
        print('No camera at all could be found from those points.')
        return
    gap, degrees = pose_error(solved, pose_b)

    fig = plt.figure(figsize=(15, 4.4))
    wide = fig.add_subplot(1, 3, 1, projection='3d')
    sample = np.random.default_rng(0).choice(len(boulder_points), 2500, replace=False)
    wide.scatter(*to_display(boulder_points[sample]).T, c=boulder_colors[sample], s=2.5,
                 alpha=0.7, linewidths=0)
    wide.plot(*to_display(camera_ring).T, color='0.6', lw=1.0)
    wide.scatter(*to_display(points_3d).T, c='orange', s=25, depthshade=False)
    draw_camera(wide, pose_b, 0.8, marker_color='k', label='the real camera')
    draw_camera(wide, solved, 0.8, marker_color='magenta', label='your camera')
    frame_around(wide, np.vstack([boulder_points, camera_ring]), margin=0.3)
    wide.view_init(elev=24, azim=-58)
    label_axes(wide)
    wide.legend(fontsize=7, loc='upper left')
    wide.set_title('the whole scene', fontsize=10)

    close = fig.add_subplot(1, 3, 2, projection='3d')
    reach = max(gap * 2.5, 0.05)
    draw_camera(close, pose_b, reach * 0.45, marker_color='k')
    draw_camera(close, solved, reach * 0.45, marker_color='magenta')
    close.plot(*to_display([pose_b['position'], solved['position']]).T,
               color='magenta', ls=':', lw=1.4)
    middle = to_display([(np.asarray(pose_b['position']) +
                          np.asarray(solved['position'])) / 2])[0]
    for setter, value in zip((close.set_xlim, close.set_ylim, close.set_zlim), middle):
        setter(value - reach, value + reach)
    close.set_box_aspect((1, 1, 1))
    close.view_init(elev=24, azim=-58)
    label_axes(close)
    close.set_title(f'zoomed in: {gap:.2f} away, aimed {degrees:.1f} degrees off',
                    fontsize=10)

    picture = fig.add_subplot(1, 3, 3)
    picture.imshow(frame_b)
    landed, _ = project(solved, points_3d, BOULDER_FOCAL, VIEW_W, VIEW_H)
    for click, spot in zip(clicks, landed):
        colour = 'orange'
        offset = spot - click
        length = np.linalg.norm(offset)
        if length > 0.3 * VIEW_W:                      # keep a wild one on screen
            offset = offset / length * 0.3 * VIEW_W
        if length > 4:
            picture.annotate('', xy=click + offset, xytext=click,
                             arrowprops=dict(arrowstyle='->', color=colour, lw=1.5))
        else:
            picture.scatter(*spot, s=22, marker='x', color=colour, linewidths=1.2)
        picture.scatter(*click, s=35, facecolors='none', edgecolors=colour, lw=1.2)
    picture.set_axis_off()
    picture.set_title(f'all {len(clicks)} points used', fontsize=10)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    show(fig)


show_solution(marked_points_3d, your_clicks[:ON_THE_ROCK],
              'the camera your clicks imply')

### Throw away your worst matches

The solve above used **all nine** of your clicks, the fumbled ones included. Nothing was
quietly ignored — which is why one bad click drags the whole answer.

`good_points` is where you get to choose. The numbers are the labels from the pictures
above; drop the ones you know you got wrong, run the cell, and watch the gap and the arrows
change.

Then try three good ones. Then three bad ones. Three is the minimum, and with three bad ones
there is nothing left to absorb the mistakes.

In [ ]:
good_points = [1, 2, 3, 4, 5, 6, 7, 8, 9]

chosen = [number - 1 for number in good_points if number <= ON_THE_ROCK]
show_solution(marked_points_3d[chosen], your_clicks[chosen],
              f'the camera from {len(chosen)} of your clicks')

## That is the whole idea

By hand, for one photo, you did what tomorrow's notebook does for every frame of your own
video: **find** points worth tracking, **match** them across photos, and **solve** for the
camera — then check by projecting the points back.

One photo gives one pose. Every frame gives a **camera track** and a cloud of 3D points,
which is exactly what the renderer needs to put something into your footage that was never
there.

# Block 3 — Putting something in

Back to the thing we parked. A photo cannot tell you how big anything is, and neither can a
pile of photos — so the reconstruction you just made has a **shape but no size**. Nothing in
that footage says whether the rock is a pebble or the size of a car.

So we do the practical thing: **guess a size, put the object in, look at it, and adjust.** A
slider for the size, your eyes for the check. That is not a workaround, it is the job — and
it is exactly what you will do tomorrow with your own video.

Before we can place anything, though, we need something to measure against.

> ## Concept — the world origin
>
> Every 3D scene needs a point that counts as **zero**: the spot all other positions are
> measured from. It is called the **origin**, and it is a choice, not a discovery — you
> put it wherever it is convenient.
>
> A sensible choice is the middle of whatever you care about. Then "up a bit and to the
> left" is a small number instead of a huge one.

> ## Concept — the normalised bounding box
>
> The **bounding box** of an object is the smallest box that contains all of it.
>
> **Normalising** means shrinking or growing everything until that box is exactly 1 unit
> across, and shifting it so its middle sits on the origin. After that, every scene —
> a rock, a room, a city — is described by numbers between about -0.5 and 0.5.
>
> Tomorrow you will write this, in a function called `compute_scene_center_and_scale`.

In [ ]:
# @title Showcase — one panda in a box
# @markdown The panda arrives at whatever size and position it happens to have. The
# @markdown switch moves the middle of its box onto the origin and scales the box down to
# @markdown exactly 1 across. Watch the numbers on the axes, not the panda -- the panda
# @markdown looks the same either way, which is the whole point.

setup_panda_scene()

normalise_switch = widgets.Checkbox(value=False, description='centre it and make the box 1 across')
out_7 = widgets.Output()


def draw_exercise_7(normalised):
    with out_7:
        out_7.clear_output(wait=True)
        corners = panda_corners()
        low, high = corners.min(0), corners.max(0)
        centre = (low + high) / 2
        width = (high - low).max()
        if normalised:
            corners = (corners - centre) / width
            low, high = corners.min(0), corners.max(0)
        # what the box measures now, after whatever we just did to it
        shown_centre = (low + high) / 2
        shown_width = (high - low).max()

        fig = plt.figure(figsize=(6.5, 5.0))
        space = fig.add_subplot(1, 1, 1, projection='3d')
        shown = to_display(corners)
        space.scatter(*shown.T, c=shown[:, 2], cmap='viridis', s=2.5, alpha=0.8,
                      linewidths=0, depthshade=False)
        draw_box(space, low, high, color='tab:orange', lw=1.2)
        space.scatter(*to_display([[0, 0, 0]]).T, color='red', s=60, label='the origin')
        frame_around(space, np.vstack([corners, [[0, 0, 0]]]),
                     margin=0.15 * float(np.ptp(corners)))
        space.view_init(elev=20, azim=-60)
        label_axes(space)
        space.legend(fontsize=8, loc='upper left')
        space.set_title('box 1 unit across, centred on the origin' if normalised
                        else 'the panda at whatever size it came in', fontsize=11)
        fig.tight_layout()
        show(fig)

        print(f'box middle    {np.round(shown_centre, 2)}')
        print(f'box width     {shown_width:.2f}')
        if normalised:
            print(f'              (moved by {np.round(-centre, 2)}, '
                  f'then divided by {width:.2f})')
        else:
            print(f'              -> to make it 1 across, divide by {width:.2f} '
                  f'(scale = {1 / width:.3f})')


display(widgets.VBox([normalise_switch, out_7]))
wire(draw_exercise_7, dict(normalised=normalise_switch))

## Giving it something to stand on

The panda is now centred on the origin — which means it is floating with its middle at
zero, half of it below. Real objects stand on something.

So we add a **ground plane**: a flat surface through the world, at some height. One dial
moves it up and down.

In [ ]:
# @title Exercise 11 — put the ground where it belongs
# @markdown Slide the ground until the panda is standing on it exactly: not sunk into it,
# @markdown not hovering above it.

ground_dial = widgets.FloatSlider(value=0.0, min=-0.8, max=0.8, step=0.01,
                                  description='ground height', continuous_update=False,
                                  layout=widgets.Layout(width='520px'))
out_8 = widgets.Output()


def draw_exercise_8(height):
    with out_8:
        out_8.clear_output(wait=True)
        show_panda_scene()
        # The dial reads the way it looks: up is positive. Y points down underneath.
        set_ground_height(-height)
        place_pandas([dict(x=0.0, y=0.0, z=0.0, scale=1.0, turn=25.0)])   # y=0 either way
        fig = plt.figure(figsize=(11, 3.6))
        for index, (title, pose) in enumerate([('from the side', PANDA_SIDE_VIEW),
                                               ('from the front', PANDA_FRONT_VIEW)]):
            panel = fig.add_subplot(1, 2, index + 1)
            panel.imshow(renderer.render_from(pose, PANDA_FOCAL))
            panel.set_title(title, fontsize=10)
            panel.set_axis_off()
        fig.tight_layout()
        show(fig)
        # Both measured the way the picture looks, with up positive: the ground sits at
        # `height`, and the panda's feet hang below the origin.
        gap = height + panda_feet_height()
        verdict = ('standing on it' if abs(gap) < 0.02
                   else 'sunk into the ground' if gap > 0 else 'floating above it')
        print(f'ground height = {height:+.2f}   ->   {verdict}')


display(widgets.VBox([ground_dial, out_8]))
wire(draw_exercise_8, dict(height=ground_dial))

## One plane, many pandas

The plane is now what everything else is measured against: say how far along the ground and
how big, and the next panda stands correctly without you thinking about heights again.

Each click of the button adds one, up to five, all arriving at the origin on top of each
other. Five dials each — three for where, one for how big, one for which way it faces. Only
one for facing, because a panda on the ground cannot lean.

**Make a family portrait.**

In [ ]:
# @title Exercise 12 — a panda family
# @markdown "add a panda" makes a new one at the origin. Spread them out along the ground,
# @markdown vary their sizes, and turn them to face wherever you like.

MAX_PANDAS = 5

add_button = widgets.Button(description='add a panda', button_style='')
family_box = widgets.VBox([])
out_9 = widgets.Output()
family_dials = []


def redraw_family(*_ignored):
    with out_9:
        out_9.clear_output(wait=True)
        show_panda_scene()
        set_ground_height(-ground_dial.value)   # the height you settled on above
        place_pandas([dict(x=row['x'].value, y=-row['y'].value, z=row['z'].value,
                           scale=row['scale'].value, turn=row['turn'].value)
                      for row in family_dials])
        fig = plt.figure(figsize=(11, 3.6))
        for index, (title, pose) in enumerate([('the shot', FAMILY_VIEW),
                                               ('from the side', PANDA_SIDE_VIEW)]):
            panel = fig.add_subplot(1, 2, index + 1)
            panel.imshow(renderer.render_from(pose, PANDA_FOCAL))
            panel.set_title(title, fontsize=10)
            panel.set_axis_off()
        fig.tight_layout()
        show(fig)


def add_panda(_button=None):
    if len(family_dials) >= MAX_PANDAS:
        return
    number = len(family_dials) + 1
    row = dict(
        x=dial(f'{number}: along', 0.0, -1.5, 1.5, step=0.05),
        z=dial(f'{number}: across', 0.0, -1.5, 1.5, step=0.05),
        y=dial(f'{number}: height', 0.0, -1.0, 1.0, step=0.02),
        scale=dial(f'{number}: size', 1.0, 0.2, 2.0, step=0.05),
        turn=dial(f'{number}: facing', 0.0, -180.0, 180.0, step=5.0))
    for slider in row.values():
        slider.observe(redraw_family, names='value')
    family_dials.append(row)
    family_box.children = list(family_box.children) + [
        widgets.HBox([row['x'], row['z'], row['y']]),
        widgets.HBox([row['scale'], row['turn']])]
    add_button.disabled = len(family_dials) >= MAX_PANDAS
    redraw_family()


add_button.on_click(add_panda)
display(widgets.VBox([add_button, family_box, out_9]))
add_panda()

## When two pandas overlap

Nothing went wrong when they stood in front of each other: the renderer knows how far away
every part of every panda is, so per pixel it keeps the nearest. Overlap is easy when you
built the scene yourself.

**A question for tomorrow.** You will put a rendered object into real footage. If it should
stand behind a real chair, what does the computer know about that chair — and what does it
not?